# 07 — Setup, FairTP Verification, R/faircause Environment

**Continuation of notebooks 01–06.** Run this notebook FIRST among the
07-13 series, top to bottom. It does three things:
1. Confirms FairTP's code is real and public.
2. Extracts and tests clean, reusable RSF/SDF functions adapted from
   FairTP's own `fsample_engine.py` — verified against synthetic data
   below.
3. Installs R + `faircause` (Plečko & Bareinboim's package) via `rpy2`.


In [1]:
# --- Clone FairTP's public repo and inspect its real structure ---
!git clone --depth 1 https://github.com/jiangnanx129/FairTP.git /content/FairTP 2>&1 | tail -5
import os
print("\nTop-level contents:")
for f in sorted(os.listdir("/content/FairTP")):
    print(" ", f)
print("\nexperiments/ contents (these are FairTP's own per-architecture folders):")
for f in sorted(os.listdir("/content/FairTP/experiments")):
    print(" ", f)


fatal: destination path '/content/FairTP' already exists and is not an empty directory.

Top-level contents:
  .git
  .gitignore
  Barchart_13_count.png
  Barchart_13_count_gba.png
  Barchart_13_count_sd.png
  HK_list_label.pkl
  HK_list_label_dcrnn_forstaticfair.pkl
  HK_list_label_dcrnn_forstaticfair_2.pkl
  HK_list_pred.pkl
  HK_list_pred_dcrnn_forstaticfair.pkl
  HK_list_pred_dcrnn_forstaticfair_2.pkl
  LICENSE
  SD_list_label.pkl
  SD_list_pred.pkl
  data
  experiments
  fsample_engine.py
  map716_region12.html
  readme.md
  region_partition.py
  region_partition_hand.py
  region_partition_hand_gba.py
  region_partition_hand_sd.py
  src
  tensors.pt

experiments/ contents (these are FairTP's own per-architecture folders):
  agcrn
  astgcn
  d2stgnn
  dcrnn
  dgcrn
  dstagnn
  gwnet


## Confirmed from this repo

- **License: MIT** — free to reuse and adapt.
- **`experiments/dcrnn/` and `experiments/gwnet/` already exist** — two of
  your four planned instruments (DCRNN, Graph WaveNet) have FairTP-compatible
  experiment scripts ready to read and adapt, across 4 dataset variants
  (HK2, HKALL, HKALLSD, SD).
- **`fsample_engine.py` contains the actual RSF/SDF formulas** (`static_cal`
  and `dynamic_cal`) — not just described in the paper, the real working
  PyTorch code. Cleaned, tested versions are below.
- No DLinear or HA implementation exists in their repo — those two
  instruments are still yours to build, matched against their DCRNN/GWN
  pattern for consistency.


In [2]:
# --- View FairTP's actual RSF formula (static_cal) directly from their repo ---
with open("/content/FairTP/fsample_engine.py") as f:
    src = f.read()
start = src.find("def static_cal")
end = src.find("def dynamic_cal")
print(src[start:end])


def static_cal(pred,label): # (b, t, 13, 1), 13表示区域数量，1表示预测的交通值维度
    b, t = pred.shape[0], pred.shape[1]
    pred = pred.reshape(b * t, -1) # 将 pred 和 label 转换为形状为 (b*t, 13)
    label = label.reshape(b * t, -1)

    # 初始化一个张量用于保存每个区域对之间的 MAPE 差值
    mape_diff = [] # torch.zeros((78,))

    # 计算 MAPE 差值
    idx = 0
    for i in range(pred.shape[1]-1): # 区域数量-1-->13-1=12
        for j in range(i + 1, pred.shape[1]):
            mape_i = torch.abs(pred[:, i] - label[:, i]) / label[:, i] # 区域i的mape
            mape_j = torch.abs(pred[:, j] - label[:, j]) / label[:, j] # 区域j的mape
            # mape_diff[idx] = torch.mean(mape_i - mape_j)
            mape_diff.append(torch.abs(torch.sum(mape_i - mape_j))) # mean换成sum
            idx += 1
        # print("--------------------:",mape_i, mape_j,mape_j.shape)

    mape_diff_mean = torch.mean(torch.stack(mape_diff), dim=0)
    return mape_diff_mean
      



In [3]:
# --- Clean, reusable RSF function, adapted from the above, tested on synthetic data ---
import numpy as np

def rsf_static_fairness(pred, label, eps=1e-6):
    """
    Region-based Static Fairness (RSF), adapted from FairTP's static_cal()
    (jiangnanx129/FairTP, fsample_engine.py).
    pred, label: arrays shaped (batch, time, n_regions)
    Returns: mean absolute pairwise MAPE-difference across all region pairs.
    Lower = fairer (regions perform more similarly to each other).
    """
    b, t, n = pred.shape
    pred = pred.reshape(b * t, n)
    label = label.reshape(b * t, n)
    mape = np.abs(pred - label) / (np.abs(label) + eps)
    diffs = []
    for i in range(n - 1):
        for j in range(i + 1, n):
            diffs.append(np.abs(np.sum(mape[:, i] - mape[:, j])))
    return float(np.mean(diffs))

# Sanity check: near-identical regions -> low RSF; one degraded region -> high RSF
rng = np.random.default_rng(0)
pred_equal = rng.normal(50, 5, size=(32, 12, 5))
label_equal = pred_equal + rng.normal(0, 0.5, size=pred_equal.shape)
print("RSF, near-identical regions:", rsf_static_fairness(pred_equal, label_equal))

pred_uneven, label_uneven = pred_equal.copy(), label_equal.copy()
label_uneven[:, :, 0] += rng.normal(0, 15, size=(32, 12))
print("RSF, one region degraded:  ", rsf_static_fairness(pred_uneven, label_uneven))
assert rsf_static_fairness(pred_uneven, label_uneven) > rsf_static_fairness(pred_equal, label_equal), \
    "Sanity check failed: degraded region should increase RSF"
print("PASS: RSF increases when a region is degraded, as expected.")


RSF, near-identical regions: 0.14522215864319957
RSF, one region degraded:   44.20486820784002
PASS: RSF increases when a region is degraded, as expected.


In [4]:
# --- Clean, reusable SDF function, tested on synthetic data ---
def sdf_dynamic_fairness(state_history_by_sensor):
    """
    Sensor-based Dynamic Fairness (SDF), adapted from FairTP's dynamic_cal().
    state_history_by_sensor: dict {sensor_id: [signed magnitudes over time]},
    positive = 'benefit' state, negative = 'sacrifice' state.
    Returns: mean absolute difference in cumulative positive/negative balance
    across all sensor pairs. Lower = fairer.
    """
    keys = list(state_history_by_sensor.keys())
    diffs = []
    for i in range(len(keys) - 1):
        for j in range(i + 1, len(keys)):
            si, sj = state_history_by_sensor[keys[i]], state_history_by_sensor[keys[j]]
            pos_i, pos_j = sum(x for x in si if x > 0), sum(x for x in sj if x > 0)
            neg_i, neg_j = sum(x for x in si if x < 0), sum(x for x in sj if x < 0)
            diffs.append(abs(pos_i - pos_j) + abs(neg_i - neg_j))
    return float(np.mean(diffs))

rng = np.random.default_rng(1)
balanced = {f"s{k}": list(rng.normal(0, 1, 50)) for k in range(6)}
print("SDF, balanced sensors:            ", sdf_dynamic_fairness(balanced))

unbalanced = {f"s{k}": list(rng.normal(0, 1, 50)) for k in range(6)}
unbalanced["s0"] = list(-np.abs(rng.normal(3, 1, 50)))
print("SDF, one sensor always sacrificed:", sdf_dynamic_fairness(unbalanced))
assert sdf_dynamic_fairness(unbalanced) > sdf_dynamic_fairness(balanced), \
    "Sanity check failed: a permanently-sacrificed sensor should increase SDF"
print("PASS: SDF increases when one sensor is chronically disadvantaged, as expected.")


SDF, balanced sensors:             4.860161175183486
SDF, one sensor always sacrificed: 60.45667536362753
PASS: SDF increases when one sensor is chronically disadvantaged, as expected.


## Now set up R — this is where `faircause` lives

Everything above was testable in this sandbox and actually ran. Everything
below (R + `faircause`) needs to be run and verified by YOU in Colab — it
could not be tested in the environment that built this notebook (no CRAN
access there). Watch for errors on first run and report back if anything
fails to install.


In [5]:
# --- Install R and rpy2 bridge (Colab has R pre-available via apt, this ensures it) ---
!apt-get install -y r-base-core -qq > /dev/null
!pip install rpy2 -q
%load_ext rpy2.ipython
print("R + rpy2 bridge loaded.")


R + rpy2 bridge loaded.


In [6]:
%%R
# Install faircause directly from GitHub (devtools needed first)
if (!requireNamespace("devtools", quietly = TRUE)) install.packages("devtools")
if (!requireNamespace("faircause", quietly = TRUE)) devtools::install_github("dplecko/CFA", upgrade = "never")
library(faircause)
cat("faircause loaded, version:", as.character(packageVersion("faircause")), "\n")

# Quick built-in-dataset sanity check (uses their own bundled census example)
data <- get(data("gov_census", package = "faircause"))
data <- as.data.frame(data[seq_len(2000), ])  # small subset, just to confirm it loads
cat("gov_census sample loaded, rows:", nrow(data), "\n")


── R CMD build ─────────────────────────────────────────────────────────────────
* checking for file ‘/tmp/RtmpaTMXRg/remotes22ae66d17c2a/dplecko-CFA-1d0dc97/DESCRIPTION’ ... OK
* preparing ‘faircause’:
* checking DESCRIPTION meta-information ... OK
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
Removed empty directory ‘faircause/vignettes’
* building ‘faircause_0.4.0.tar.gz’

Installing 3 packages: xgboost, grf, ranger
Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/xgboost_3.2.1.1.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/grf_2.6.1.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/ranger_0.18.0.tar.gz'

The downloaded source packages are in
	‘/tmp/RtmpaTMXRg/downloaded_packages’
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
Error in library(faircause) : there is no package 

RInterpreterError: Failed to parse and evaluate line '# Install faircause directly from GitHub (devtools needed first)\nif (!requireNamespace("devtools", quietly = TRUE)) install.packages("devtools")\nif (!requireNamespace("faircause", quietly = TRUE)) devtools::install_github("dplecko/CFA", upgrade = "never")\nlibrary(faircause)\ncat("faircause loaded, version:", as.character(packageVersion("faircause")), "\\n")\n\n# Quick built-in-dataset sanity check (uses their own bundled census example)\ndata <- get(data("gov_census", package = "faircause"))\ndata <- as.data.frame(data[seq_len(2000), ])  # small subset, just to confirm it loads\ncat("gov_census sample loaded, rows:", nrow(data), "\\n")\n'.
R error message: 'Error in library(faircause) : there is no package called ‘faircause’'
R stdout:
Downloading GitHub repo dplecko/CFA@HEAD
Installing 3 packages: xgboost, grf, ranger
Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstu

## What to do if the R cell above fails

`faircause` is explicitly noted by its authors as still under development
(v0.2.0). If `devtools::install_github("dplecko/CFA")` fails:
1. Check the error message for a missing system dependency (common ones:
   `libcurl4-openssl-dev`, `libssl-dev` — install via
   `!apt-get install -y libcurl4-openssl-dev libssl-dev` then retry).
2. Check the repo's Issues page (github.com/dplecko/CFA/issues) for the
   specific error — it's a small, actively maintained package, so install
   issues are usually already reported there.
3. As a fallback, the SFM projection logic and Ctf-DE/IE/SE formulas are
   documented in the paper itself (Plečko & Bareinboim 2024) if you need to
   verify a computation by hand while the package issue gets resolved.
